# Chapter 08: Miscellaneous 2D Problems

Source orientation: printed pages 285-324; PDF pages 332-371.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. The goal is not to reproduce the book; the goal is to turn the chapter into an inspectable graphics-geometry lab. A reader should be able to learn the main ideas here with no PDF open.

## Chapter Goal

Circle and line constructions with tangency, prescribed radius, offsets, perpendicularity, and solution multiplicity.

Geometric tools for graphics are easiest to remember when each formula is connected to a representation, a picture, and a check. This lesson therefore treats the chapter as a sequence of decisions a programmer makes: how to represent the primitive, which parameters define the query, what degeneracies are possible, and how to verify that the computed answer is still geometric. The same discipline applies to renderers, modeling tools, collision systems, curve editors, CAD importers, and mesh-processing code. Throughout the notebook, formulas are rewritten as small executable experiments so that the main concept can be rotated, sampled, plotted, and tested.

The miscellaneous 2D constructions chapter is about reducing constraints to loci. The notebook shows how tangency becomes an offset-line or offset-circle problem, and why solution count changes at exact threshold distances. The checks focus on equal distance and perpendicular radius conditions because those are the invariants every construction should satisfy.

## Translation Guide

- **Representation:** choose arrays, frames, equations, graphs, or meshes that make Miscellaneous 2D Problems inspectable.
- **Domain:** identify whether parameters are free, clamped, periodic, barycentric, bounded by a simplex, or constrained by a surface.
- **Invariant:** decide what should not change under translation, rotation, reparameterization, input order, or coordinate-system choice.
- **Residual:** convert the invariant into a number that can be asserted after the figure is drawn.
- **Failure mode:** expose degeneracy, near-zero denominators, endpoint cases, tangencies, or rank loss instead of hiding them behind a single Boolean answer.

This guide is intentionally repeated across the course because it is the working style that makes a reference book become a usable notebook course. Each chapter changes the objects and algorithms, but the learning loop remains the same: model, draw, perturb, measure, and check.

## Route Through The Chapter

1. Translate the chapter vocabulary into computational objects.
2. Build visual artifacts that expose the main invariants.
3. Run a small numeric experiment that makes stability or classification visible.
4. Close with sanity checks that make the notebook reproducible.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the GTCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-08"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

## Visual Storyboard

- circle through three points
- offset-line tangent construction
- two-circle tangent families
- solution-count phase diagram

Each planned visual names the geometric behavior to inspect, not just the rendering technology. Static PNG artifacts are used for durable diagrams. If future work adds interactive widgets or Plotly scenes, they should still write stable HTML artifacts under the same chapter artifact subtree.

## Core Concepts

### 1. Construction problems are solved by translating constraints into simpler loci

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **circle through three points**, so the reader can inspect the construction rather than only read a formula. The paired check is **constructed circle centers are equidistant from constraints**, which turns the claim into an executable invariant. In Miscellaneous 2D Problems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

### 2. Circle tangency often becomes intersection of offset lines or offset circles

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **offset-line tangent construction**, so the reader can inspect the construction rather than only read a formula. The paired check is **tangent radius is perpendicular to the tangent line at contact**, which turns the claim into an executable invariant. In Miscellaneous 2D Problems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

### 3. The number of solutions changes at degenerate threshold distances

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **two-circle tangent families**, so the reader can inspect the construction rather than only read a formula. The paired check is **solution counts match distance thresholds**, which turns the claim into an executable invariant. In Miscellaneous 2D Problems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

### 4. A robust implementation must return zero, one, two, or four solutions without surprise

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **solution-count phase diagram**, so the reader can inspect the construction rather than only read a formula. The paired check is **perpendicular and parallel constructed lines satisfy dot-product tests**, which turns the claim into an executable invariant. In Miscellaneous 2D Problems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

## Worked Example Pattern

The worked example below uses compact synthetic data rather than copied textbook figures. That is deliberate: a synthetic example can be regenerated, perturbed, and checked. The first artifact is a concept map that connects the chapter goal to the planned visuals. The second artifact is a geometric scene specialized to Miscellaneous 2D Problems. The third artifact is a numeric diagnostic that turns a qualitative claim into a curve or residual. When a chapter contains many algorithms, this pattern becomes a template for further exploration: choose a primitive, construct a small query, draw the active features, then assert the invariants that justify the returned answer. The examples are small enough to read but structured enough that a learner can swap in their own points, matrices, curves, surfaces, or meshes.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np

from utils.artifacts import display_artifact, save_json, save_matplotlib
from utils.chapter_visuals import compute_check_values, concept_map_figure, geometry_scene_figure, numerical_experiment_figure, storyboard_gallery_figure
from utils.validation import artifact_report, require_nonempty

ENTRY_TITLE = 'Miscellaneous 2D Problems'
MODE = 'misc2d'
TOPIC = 'chapter-08'
CONCEPTS = ['construction problems are solved by translating constraints into simpler loci', 'circle tangency often becomes intersection of offset lines or offset circles', 'the number of solutions changes at degenerate threshold distances', 'a robust implementation must return zero, one, two, or four solutions without surprise']
VISUALS = ['circle through three points', 'offset-line tangent construction', 'two-circle tangent families', 'solution-count phase diagram']
CHECKS = ['constructed circle centers are equidistant from constraints', 'tangent radius is perpendicular to the tangent line at contact', 'solution counts match distance thresholds', 'perpendicular and parallel constructed lines satisfy dot-product tests']
SEED = 8
artifact_paths = []

In [ ]:
fig = concept_map_figure(ENTRY_TITLE, CONCEPTS, VISUALS)
concept_map_path = save_matplotlib(fig, TOPIC, "figures", "miscellaneous-2d-problems-concept-route-map.png")
plt.close(fig)
artifact_paths.append(concept_map_path)
display_artifact(concept_map_path, width=820)

In [ ]:
fig = storyboard_gallery_figure(MODE, ENTRY_TITLE, VISUALS, SEED)
storyboard_gallery_path = save_matplotlib(fig, TOPIC, "figures", "storyboard-gallery.png")
plt.close(fig)
artifact_paths.append(storyboard_gallery_path)
display_artifact(storyboard_gallery_path, width=820)

In [ ]:
fig = geometry_scene_figure(MODE, ENTRY_TITLE, SEED)
geometry_scene_path = save_matplotlib(fig, TOPIC, "figures", "miscellaneous-2d-problems-construction-scene.png")
plt.close(fig)
artifact_paths.append(geometry_scene_path)
display_artifact(geometry_scene_path, width=820)

## Applied Lab

Use the code cells as a starting point, then replace the supplied sample data with a case from your own graphics or geometry pipeline. For this chapter, vary one primitive, one tolerance, and one coordinate frame. Record which visual changes are geometric and which are artifacts of representation. A good lab notebook for Miscellaneous 2D Problems should include the input data, the rendered artifact path, the numeric residual, and a one-paragraph explanation of what would count as a failure in production code.

Suggested extension: build a second example that is nearly degenerate. Move one point close to a boundary, make two axes almost parallel, set a polynomial root almost double, or shrink a determinant toward zero. Then compare the visual artifact with the numeric check. The point of the lab is not to memorize a formula; it is to practice recognizing when a geometric answer is trustworthy, underdetermined, unstable, or dependent on a convention.

## Sanity Checklist

- constructed circle centers are equidistant from constraints
- tangent radius is perpendicular to the tangent line at contact
- solution counts match distance thresholds
- perpendicular and parallel constructed lines satisfy dot-product tests

The final code cell writes `final-sanity.json` into the chapter artifact subtree. The JSON is intentionally small: it records artifact sizes and a few chapter-specific numeric values so that later audits can distinguish a real teaching artifact from a blank or decorative image.

In [ ]:
fig = numerical_experiment_figure(MODE, ENTRY_TITLE, SEED)
numeric_diagnostic_path = save_matplotlib(fig, TOPIC, "figures", "numeric-diagnostic.png")
plt.close(fig)
artifact_paths.append(numeric_diagnostic_path)
display_artifact(numeric_diagnostic_path, width=820)

In [ ]:
check_values = compute_check_values(MODE)
assert check_values["max_error"] <= check_values["tolerance"], check_values
check_values

In [ ]:
require_nonempty(artifact_paths, min_bytes=1500)
final_sanity = {
    "topic": TOPIC,
    "title": ENTRY_TITLE,
    "mode": MODE,
    "artifacts": artifact_report(artifact_paths, root=BOOK_ROOT),
    "check_values": check_values,
    "checks": CHECKS,
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert sanity_path.exists() and sanity_path.stat().st_size > 200
final_sanity

## Takeaways

- Miscellaneous 2D Problems is a set of geometric modeling choices, not merely a list of formulas.
- The durable learning object is the combination of prose, figure, executable construction, and residual.
- Book-local artifacts make the notebook reproducible and reviewable without embedding large outputs directly in the notebook.
- A useful graphics-geometry implementation reports enough state to debug degeneracy, tolerance, and convention errors.